# 📊 Notebook 02 — Análise Espacial
**Projeto:** Sinop Agro-GIS — Expansão Agrícola em Mato Grosso  
**Autor:** Jakson Pascoal | github.com/Jk-Pascoal  
**Dados:** IBGE Malha Municipal 2022 + MapBiomas Collection 10.1  

---

## Objetivos deste notebook
1. Carregar e explorar o shapefile do IBGE (municípios de MT)
2. Filtrar e isolar o município de Sinop (IBGE: 5107909)
3. Calcular estatísticas básicas (área, perímetro, centroide)
4. Analisar os dados de uso do solo MapBiomas
5. Gerar tabela comparativa de uso do solo por classe
6. Exportar shapefile recortado de Sinop

## 0. Imports e Configurações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# Caminhos do projeto
ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'data'
MAPS_DIR  = ROOT / 'maps' / 'exportados'

print('✅ Imports OK')
print(f'📁 Raiz do projeto: {ROOT}')
print(f'📁 Dados: {DATA_DIR}')

## 1. Carregar Shapefile do IBGE — Municípios de MT

In [ ]:
# Localizar o shapefile
shp_path = DATA_DIR / 'limite_municipal' / 'MT_Municipios_2022.shp'

if not shp_path.exists():
    print(f'❌ Shapefile não encontrado em: {shp_path}')
    print('   Execute: python scripts/download_dados.py')
else:
    gdf_mt = gpd.read_file(shp_path)
    print(f'✅ Shapefile carregado: {len(gdf_mt)} municípios em MT')
    print(f'   CRS: {gdf_mt.crs}')
    print(f'   Colunas: {list(gdf_mt.columns)}')
    gdf_mt.head(3)

In [ ]:
# Filtrar apenas Sinop (código IBGE: 5107909)
CODIGO_SINOP = '5107909'

# Verificar qual coluna tem o código
col_codigo = [c for c in gdf_mt.columns if 'CD' in c.upper() or 'COD' in c.upper()][0]
print(f'Coluna de código: {col_codigo}')

sinop = gdf_mt[gdf_mt[col_codigo] == CODIGO_SINOP].copy()

if sinop.empty:
    # Tentar com int
    sinop = gdf_mt[gdf_mt[col_codigo] == int(CODIGO_SINOP)].copy()

print(f'\n✅ Sinop encontrada: {len(sinop)} feição(ões)')
print(sinop[['NM_MUN', col_codigo, 'AREA_KM2']].to_string() if 'AREA_KM2' in sinop.columns else sinop.head())

## 2. Estatísticas Geométricas de Sinop

In [ ]:
# Reprojetar para SIRGAS 2000 UTM Zone 21S (EPSG:31981) para cálculos métricos
sinop_utm = sinop.to_crs(epsg=31981)

area_km2     = sinop_utm.geometry.area.iloc[0] / 1e6
area_ha      = area_km2 * 100
perimetro_km = sinop_utm.geometry.length.iloc[0] / 1e3
centroide    = sinop.geometry.centroid.iloc[0]

print('=' * 50)
print('📍 SINOP — Estatísticas Geográficas')
print('=' * 50)
print(f'  Área:        {area_km2:,.1f} km²')
print(f'  Área:        {area_ha:,.0f} ha')
print(f'  Perímetro:   {perimetro_km:,.1f} km')
print(f'  Centróide:   Lon {centroide.x:.4f}°, Lat {centroide.y:.4f}°')
print(f'  CRS (orig):  {sinop.crs}')
print('=' * 50)

## 3. Visualização — Mapa de Sinop no contexto de MT

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#0a0a0f')

# ── Mapa 1: MT completo com Sinop destacado ──────────────────
ax1 = axes[0]
ax1.set_facecolor('#0a0a0f')

gdf_mt.plot(ax=ax1, color='#1a2a1a', edgecolor='#2a4a2a', linewidth=0.4)
sinop.plot(ax=ax1, color='#4ade80', edgecolor='#22c55e', linewidth=1.5)

ax1.set_title('Sinop no contexto de\nMato Grosso', color='#e0e0e0',
              fontsize=12, fontweight='bold', pad=10)
ax1.axis('off')

# Legenda
patches = [
    mpatches.Patch(color='#1a2a1a', label='Outros municípios MT'),
    mpatches.Patch(color='#4ade80', label='Sinop'),
]
ax1.legend(handles=patches, loc='lower left',
           facecolor='#111', edgecolor='#333',
           labelcolor='#ccc', fontsize=9)

# ── Mapa 2: Sinop isolado ────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0a0a0f')

sinop.plot(ax=ax2, color='#166534', edgecolor='#4ade80', linewidth=2)

cx = centroide.x
cy = centroide.y
ax2.annotate(f'Sinop\n{area_km2:,.0f} km²',
             xy=(cx, cy), fontsize=11, color='#fff',
             fontweight='bold', ha='center', va='center')

ax2.set_title('Município de Sinop-MT\n(IBGE 2022 — SIRGAS 2000)',
              color='#e0e0e0', fontsize=12, fontweight='bold', pad=10)
ax2.axis('off')

plt.suptitle('Análise Espacial — Sinop, Mato Grosso',
             color='#fff', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

out = MAPS_DIR / '06_sinop_contexto_mt.png'
plt.savefig(out, dpi=150, bbox_inches='tight',
            facecolor='#0a0a0f', edgecolor='none')
plt.show()
print(f'✅ Salvo: {out}')

## 4. Análise de Uso do Solo — MapBiomas 2024
> Dados consolidados da Collection 10.1 para o município de Sinop

In [ ]:
# Dados do MapBiomas Collection 10.1 — Sinop 2024
# Fonte: maps/exportados/02_uso_solo_sinop_2024.png + uso_solo_sinop_2024.html

TOTAL_HA = 399_085.86

uso_solo = pd.DataFrame([
    # Categoria,          Classe,                Área (ha),    Cor MapBiomas
    ('Floresta',          'Floresta Nativa',      129_179.01,   '#1F8D49'),
    ('Floresta',          'Floresta Aluvial',      15_470.66,   '#519799'),
    ('Floresta',          'Savana Arbórea',              2.63,  '#7DC975'),
    ('Herb./Arbustiva',   'Campo Natural',             31.08,   '#d6bc74'),
    ('Herb./Arbustiva',   'Área Úmida',             1_306.50,   '#45C2A5'),
    ('Agropecuária',      'Soja',                 169_851.83,   '#E974ED'),
    ('Agropecuária',      'Algodão',                   122.93,  '#c27ba0'),
    ('Agropecuária',      'Outras Lavouras',        28_619.36,  '#e8c3e8'),
    ('Agropecuária',      'Plantio Florestal',         410.34,  '#7a5c58'),
    ('Agropecuária',      'Pastagem',               30_132.11,  '#BBB35A'),
    ('Área não veg.',     'Área Urbana',             8_938.24,  '#DB4040'),
    ('Área não veg.',     'Outras Áreas',            1_635.16,  '#b0a090'),
    ('Água',              'Rios e Lagos',            13_386.03,  '#2532E4'),
], columns=['categoria', 'classe', 'area_ha', 'cor'])

uso_solo['area_pct'] = (uso_solo['area_ha'] / TOTAL_HA * 100).round(2)

# Sumário por categoria
por_categoria = (uso_solo
    .groupby('categoria')[['area_ha']]
    .sum()
    .assign(area_pct=lambda df: (df['area_ha'] / TOTAL_HA * 100).round(1))
    .sort_values('area_ha', ascending=False)
)

print('=' * 55)
print('🌍 USO DO SOLO — SINOP-MT 2024 (MapBiomas Col. 10.1)')
print('=' * 55)
print(f'{'Categoria':<20} {'Área (ha)':>12} {'%':>8}')
print('-' * 42)
for cat, row in por_categoria.iterrows():
    print(f'{cat:<20} {row.area_ha:>12,.0f} {row.area_pct:>7.1f}%')
print('-' * 42)
print(f'{'TOTAL':<20} {TOTAL_HA:>12,.0f} {100.0:>7.1f}%')
print('=' * 55)

In [ ]:
# Tabela detalhada por classe
print('\n📋 DETALHAMENTO POR CLASSE:')
print(uso_solo[['categoria','classe','area_ha','area_pct']]
      .sort_values('area_ha', ascending=False)
      .to_string(index=False, float_format='{:,.2f}'.format))

## 5. Insight Principal — Razão Floresta / Agropecuária

In [ ]:
area_floresta = uso_solo[uso_solo['categoria']=='Floresta']['area_ha'].sum()
area_agro     = uso_solo[uso_solo['categoria']=='Agropecuária']['area_ha'].sum()
area_soja     = uso_solo[uso_solo['classe']=='Soja']['area_ha'].iloc[0]
area_urbana   = uso_solo[uso_solo['classe']=='Área Urbana']['area_ha'].iloc[0]

print('\n🔍 INSIGHTS — Sinop-MT 2024')
print('='*55)
print(f'  • Ratio Agro/Floresta:     {area_agro/area_floresta:.2f}x')
print(f'    → Para cada 1 ha de floresta, há {area_agro/area_floresta:.1f} ha de agropecuária')
print()
print(f'  • Soja sobre área total:   {area_soja/TOTAL_HA*100:.1f}%')
print(f'    → 42 de cada 100 ha do município = soja')
print()
print(f'  • Área urbana é apenas:    {area_urbana/TOTAL_HA*100:.1f}% do território')
print(f'    → Cidade com ~150k hab. ocupa menos que rios e lagos')
print()
print(f'  • Floresta restante:       {area_floresta:,.0f} ha ({area_floresta/TOTAL_HA*100:.1f}%)')
print(f'    → Equivalente a ~3x a área do município de São Paulo')
print('='*55)

## 6. Exportar Shapefile de Sinop recortado

In [ ]:
out_shp = DATA_DIR / 'limite_municipal' / 'sinop_recorte.shp'

if not sinop.empty:
    sinop.to_file(out_shp)
    print(f'✅ Shapefile de Sinop exportado: {out_shp}')
    print(f'   CRS: {sinop.crs}')
    print(f'   Feições: {len(sinop)}')
else:
    print('❌ sinop GeoDataFrame está vazio — verifique o shapefile do IBGE')

---
## ✅ Resumo do Notebook 02

| Etapa | Status |
|---|---|
| Shapefile IBGE carregado | ✅ |
| Sinop filtrado (IBGE 5107909) | ✅ |
| Estatísticas geométricas calculadas | ✅ |
| Mapa de contexto exportado | ✅ |
| Tabela de uso do solo (MapBiomas) | ✅ |
| Insights calculados | ✅ |
| Shapefile recortado exportado | ✅ |

**Próximo:** `03_visualizacao.ipynb` — Gráfico temporal 2000–2024 + Mapa Folium interativo